# Business Entity Resolution — Implementation Notebook

**Runs on:** SageMaker notebook instance `ml.t3.medium` (2 vCPU / 4 GB RAM, AL2023, JupyterLab), `conda_python3` kernel.
**Rule of this notebook:** everything executable runs in **SAMPLE mode** (5k-S1 skeleton, ≤15k-doc ANN demo), which fits 4 GB RAM. The full-data path (2.2M S1 × 10M candidates) ships as `RUN_FULL` stubs that refuse to run on `t3.medium` and tell you which instance to use instead (see *Phase 2 scale-up*).
**Plan ref:** `../ENTITY_RESOLUTION_PLAN.md` — Phase 0 → 0.5 → 1 → 2 (sample) → 3/4/5 (roadmap + stubs). All `.tsv` reads use `sep="\t"`.

Expected data layout (upload once to the instance, e.g. `/home/ec2-user/SageMaker/`):
```
<DATA_ROOT>/train/train_source1.tsv
<DATA_ROOT>/train/train_source2.tsv
<DATA_ROOT>/train/train_source3.tsv
<DATA_ROOT>/train/train_ground_truth.tsv
<DATA_ROOT>/test/test_source1.tsv ...
```

Colab option: dataset at Drive `MyDrive/AmazonMLChallenge/dataset` is auto-copied to `/content/dataset` by the Drive cell below.


## Which cells to run / skip (read this first)

**First run:** Run All, but expect exactly ONE red cell - the `RUN_FULL` assert near the bottom. It raises `AssertionError` by design on `t3.medium` (4 GB cannot index ~10M docs). A red `RUN_FULL` cell means the guardrail works, not that anything broke.

| Cell | Verdict |
|---|---|
| `%pip install` | Run once per instance - SageMaker persists the env, skip on re-runs |
| Config, scorer + unit test, val split, normalization lib | Must run every time |
| Skeleton run (PIN block, score, threshold) | Must run - the core pipeline |
| TSV write + validation | Must run - must print `PASS` |
| France asserts | Optional - instant sanity check, safe to skip |
| FAISS ANN demo | Optional - index smoke test only (PIN-biased pool, not a recall claim) |
| `RUN_FULL` assert | **Do NOT run** - fails by design until the full recipe is implemented |
| Summary print | Optional - prints sample stats + output paths |

In [ ]:
%pip install -q faiss-cpu rapidfuzz lightgbm

In [ ]:
from pathlib import Path

# ---- Optional: Google Drive -> fast local workspace copy (Colab only, no-op elsewhere) ----
# Dataset on Drive: MyDrive/AmazonMLChallenge/dataset/{train,test}/
# On SageMaker ignore this cell (upload dataset/ beside the notebook instead);
# on Colab it populates /content/dataset, which the config cell below auto-detects.
WORKSPACE_DATASET = Path("/content/dataset")
try:
    from google.colab import drive
    drive.mount("/content/drive", force_remount=False)
    src = Path("/content/drive/MyDrive/AmazonMLChallenge/dataset")
    want = ["train/train_source1.tsv", "test/test_source1.tsv"]
    if all((WORKSPACE_DATASET / f).exists() for f in want):
        print("workspace copy already present:", WORKSPACE_DATASET)
    elif all((src / f).exists() for f in want):
        print("copying Drive dataset to workspace SSD (~2.5 GB, one-time)...")
        WORKSPACE_DATASET.mkdir(parents=True, exist_ok=True)
        import subprocess
        r = subprocess.run(["rsync", "-ah", "--info=progress2", str(src) + "/", str(WORKSPACE_DATASET) + "/"])
        if r.returncode != 0:
            import shutil
            shutil.copytree(src, WORKSPACE_DATASET, dirs_exist_ok=True)
        print("copy done")
    else:
        print("Drive dataset not found at", src)
except ImportError:
    print("not on Colab - upload dataset/ beside this notebook (SageMaker: file browser or S3).")


In [ ]:
from pathlib import Path
import hashlib
import re
import resource
import time
import unicodedata

import numpy as np
import pandas as pd

# ---- config ----
CANDIDATE_ROOTS = [Path.cwd(), Path.cwd().parent,
                   Path("/home/ec2-user/SageMaker"), Path.home()]
DATA_ROOT = None
for r in CANDIDATE_ROOTS:
    for c in [r / "dataset",
              r / "student_resource" / "student_resource" / "dataset",
              r / "student_resource" / "dataset"]:
        if (c / "train" / "train_source1.tsv").exists():
            DATA_ROOT = c
            break
    if DATA_ROOT is not None:
        break
print("DATA_ROOT =", DATA_ROOT)

SAMPLE_S1 = 5000        # skeleton queries (fits t3.medium)
POOL_DOCS = 15000       # ANN demo pool cap (dense 4096-dim ~= 245 MB)
TOP_K = 10
RANDOM_STATE = 42
RUN_FULL = False        # True only on a large instance (see scale-up cell)
OUT_DIR = Path("output"); OUT_DIR.mkdir(exist_ok=True)

assert DATA_ROOT is not None, ("Dataset not found — copy dataset/train + dataset/test "
    "next to this notebook and re-run.")

## Phase 0 — Macro-F0.5 scorer (with unit test)
Singleton semantics: empty/empty → 1.0; any false merge on a singleton → 0.0.

In [ ]:
def f05_single(y_true: set, y_pred: set) -> float:
    tp = len(y_true & y_pred)
    prec = tp / len(y_pred) if y_pred else (1.0 if not y_true else 0.0)
    rec = tp / len(y_true) if y_true else (1.0 if not y_pred else 0.0)
    if prec + rec == 0:
        return 0.0
    return 1.25 * prec * rec / (0.25 * prec + rec)


def macro_f05(truth: dict, pred: dict) -> float:
    return sum(f05_single(set(truth[k]), set(pred.get(k, ()))) for k in truth) / len(truth)


# unit tests: PDF worked example S1-00001 -> P=2/3, R=1.0, F0.5=0.7142857
assert abs(f05_single({"S2-00047", "S3-00812"}, {"S2-00047", "S2-00193", "S3-00812"}) - 0.7142857) < 1e-6
assert f05_single(set(), set()) == 1.0
assert f05_single(set(), {"S2-1"}) == 0.0
assert f05_single({"S2-1"}, set()) == 0.0
print("scorer OK: pdf-example=0.7142857, singleton-empty=1.0, singleton-fp=0.0")

## Phase 0 — Validation split (hash-based, single chunked pass)
Deterministic 10% of S1 ids (`md5 % 10 == 0`), stratified *reporting* by match-count bucket. No full-file load: GT streams in chunks, only val rows are kept (~220k).

In [ ]:
def in_val(s1: str) -> bool:
    return hashlib.md5(s1.encode()).digest()[0] % 10 == 0


def bucket(n: int) -> str:
    if n == 0:
        return "0-singleton"
    if n == 1:
        return "1"
    if n <= 3:
        return "2-3"
    if n <= 5:
        return "4-5"
    return "6+"


val_matches: dict = {}
total = 0
t0 = time.time()
for ch in pd.read_csv(DATA_ROOT / "train" / "train_ground_truth.tsv", sep="\t", chunksize=200000):
    s1s = ch["source1_entity_id"].astype(str)
    ms = ch["matched_entity_ids"].fillna("").astype(str)
    for s1, m in zip(s1s, ms):
        total += 1
        if in_val(s1):
            m = m.strip()
            val_matches[s1] = [x for x in m.split(",") if x] if m else []
print(f"GT rows={total} val_S1={len(val_matches)} ({len(val_matches)/total:.1%}) in {time.time()-t0:.0f}s")
dist: dict = {}
for v in val_matches.values():
    b = bucket(len(v))
    dist[b] = dist.get(b, 0) + 1
print("val bucket distribution:", dist)
print("val singleton rate:", round(dist.get('0-singleton', 0) / len(val_matches), 4))

## Phase 0.5 — Walking skeleton (trivial PIN blocking, full pipeline, 5k S1)
Purpose is *integration*, not recall: prove load → normalize → block → features → score → TSVs → validator PASS on `t3.medium` before building real blocking.

In [ ]:
US_IN_ABBR = {"corp": "corporation", "inc": "incorporated", "pvt": "private",
               "ltd": "limited", "rd": "road", "st": "street", "ave": "avenue",
               "blvd": "boulevard", "ste": "suite", "apt": "apartment",
               "mfg": "manufacturing", "ent": "enterprises", "co": "company"}
FR_ABBR = {"sarl": "societe responsabilite limitee", "sas": "societe actions simplifiee",
           "sa": "societe anonyme", "eurl": "entreprise unipersonnelle",
           "rue": "rue", "bd": "boulevard", "av": "avenue", "pl": "place",
           "imp": "impasse", "cedex": "cedex", "ste": "societe", "ets": "etablissements"}
ABBR = {**US_IN_ABBR, **FR_ABBR}  # open-set: FR included despite 0% train coverage
LEGAL_SUFFIX = set(ABBR) | {"corporation", "incorporated", "private", "limited",
                "company", "llc", "llp", "gmbh", "societe"}
PIN_RE = r"(?<!\d)(\d{5,6})(?!\d)"  # IN 6-digit, US/FR 5-digit


def normalize_text(s: str) -> str:
    s = unicodedata.normalize("NFKD", str(s)).encode("ascii", "ignore").decode()
    s = s.lower().replace("&", " and ")
    s = re.sub(r"[^a-z0-9 ]", " ", s)
    toks = [ABBR.get(t, t) for t in s.split()]
    return re.sub(r"\s+", " ", " ".join(toks)).strip()


def extract_pin(addr: str) -> str:
    m = re.search(PIN_RE, str(addr))
    return m.group(1) if m else ""


def suffix_match(a: str, b: str) -> int:
    ta, tb = a.split(), b.split()
    if not ta or not tb:
        return 0
    return int((ta[-1] in LEGAL_SUFFIX) == (tb[-1] in LEGAL_SUFFIX))


print("Lumay Bóral ->", normalize_text("Lumay Bóral"))
print("SARL Dupont, 12 Rue de la Paix, 75002 Paris ->", normalize_text("SARL Dupont, 12 Rue de la Paix, 75002 Paris"))
print("PIN:", extract_pin("1056c Belden Ave, AKON, Ohio 44308"), extract_pin("75002 Paris, Cedex"))

### Skeleton run: PIN-block → 3-feature score → threshold → TSVs → checks

In [ ]:
from rapidfuzz import fuzz

sample_s1 = sorted(val_matches)[:SAMPLE_S1]
sample_set = set(sample_s1)
print(f"skeleton queries: {len(sample_s1)}")

# 1. load sample S1 rows (chunked scan, keep targets only)
s1_parts = []
for ch in pd.read_csv(DATA_ROOT / "train" / "train_source1.tsv", sep="\t", chunksize=200000):
    hit = ch[ch["entity_id"].astype(str).isin(sample_set)]
    if len(hit):
        s1_parts.append(hit)
s1_df = pd.concat(s1_parts, ignore_index=True)
assert len(s1_df) == len(sample_s1), (len(s1_df), len(sample_s1))
s1_df["norm_name"] = s1_df["business_name"].fillna("").map(normalize_text)
s1_df["norm_addr"] = s1_df["business_address"].fillna("").map(normalize_text)
s1_df["pin"] = s1_df["business_address"].fillna("").map(extract_pin)
pin_to_s1: dict = {}
for r in s1_df.itertuples():
    if r.pin:
        pin_to_s1.setdefault(r.pin, []).append(r.entity_id)
print(f"sample S1 with PIN: {(s1_df['pin'] != '').sum()} / {len(s1_df)}")

# 2. PIN-blocked pool scan over S2+S3 (vectorized extract per chunk)
pool_parts = []
for name in ["train_source2.tsv", "train_source3.tsv"]:
    for ch in pd.read_csv(DATA_ROOT / "train" / name, sep="\t", chunksize=200000):
        pins = ch["business_address"].fillna("").str.extract(PIN_RE, expand=False)
        hit = ch[pins.isin(set(pin_to_s1))]
        if len(hit):
            pool_parts.append(hit)
pool_df = pd.concat(pool_parts, ignore_index=True).drop_duplicates("entity_id") if pool_parts else s1_df.iloc[0:0].copy()
pool_df["norm_name"] = pool_df["business_name"].fillna("").map(normalize_text)
pool_df["norm_addr"] = pool_df["business_address"].fillna("").map(normalize_text)
pool_df["pin"] = pool_df["business_address"].fillna("").map(extract_pin)
print(f"pool rows sharing a PIN with sample: {len(pool_df)}")

# 3. candidates: every pool row sharing the S1 PIN (cap 40 = sanity bound)
pool_by_pin: dict = {}
for r in pool_df.itertuples():
    pool_by_pin.setdefault(r.pin, []).append(r.entity_id)
pool_idx = {r.entity_id: r for r in pool_df.itertuples()}
s1_idx = {r.entity_id: r for r in s1_df.itertuples()}
candidates = {s: list(pool_by_pin.get(s1_idx[s].pin, []))[:40] for s in sample_s1}
mean_k = float(np.mean([len(v) for v in candidates.values()]))
print(f"mean K={mean_k:.1f}, S1 with >=1 candidate={(sum(1 for v in candidates.values() if v))}/{len(candidates)}")

# 4. score (skeleton: untrained weighted sum) + threshold -> matches
def pair_score(a, b) -> float:
    ratio = fuzz.WRatio(a.norm_name, b.norm_name) / 100.0
    ta, tb = set(a.norm_name.split()), set(b.norm_name.split())
    jac = len(ta & tb) / max(1, len(ta | tb))
    pin = 1.0 if (a.pin and a.pin == b.pin) else 0.0
    return 0.5 * ratio + 0.3 * jac + 0.2 * pin


TAU_SKELETON = 0.5
matches = {}
for s in sample_s1:
    a = s1_idx[s]
    scored = sorted(((pool_idx[c], pair_score(a, pool_idx[c])) for c in candidates[s]),
                    key=lambda t: -t[1])
    matches[s] = [r.entity_id for r, sc in scored if sc >= TAU_SKELETON]
skel_f05 = macro_f05({s: val_matches[s] for s in sample_s1}, matches)
print(f"skeleton macro-F0.5 on sample (reference only, PIN-blocked): {skel_f05:.4f}")

In [ ]:
# 5. write TSVs (tab-separated, utf-8) + validate
def write_id_list(path: Path, rows: dict, col: str):
    df = pd.DataFrame({"source1_entity_id": list(rows.keys()),
                       col: [",".join(rows[k]) for k in rows]})
    df.to_csv(path, sep="\t", index=False, encoding="utf-8")


write_id_list(OUT_DIR / "matching_results.tsv", matches, "matched_entity_ids")
write_id_list(OUT_DIR / "candidate_pairs.tsv", candidates, "candidate_entity_ids")
print("wrote", OUT_DIR / "matching_results.tsv", "and", OUT_DIR / "candidate_pairs.tsv")


def check_outputs(match_path: Path, cand_path: Path, required: set):
    issues = []
    m = pd.read_csv(match_path, sep="\t", keep_default_na=False)
    c = pd.read_csv(cand_path, sep="\t", keep_default_na=False)
    if list(m.columns) != ["source1_entity_id", "matched_entity_ids"]:
        issues.append(f"matching header: {list(m.columns)}")
    if list(c.columns) != ["source1_entity_id", "candidate_entity_ids"]:
        issues.append(f"candidate header: {list(c.columns)}")
    if set(m["source1_entity_id"]) != required or len(m) != len(required):
        issues.append("matching S1 row coverage mismatch")
    if set(c["source1_entity_id"]) != required or len(c) != len(required):
        issues.append("candidate S1 row coverage mismatch")
    valid = set(pool_df["entity_id"].astype(str))
    cmap = {}
    for _, r in c.iterrows():
        ids = [x for x in str(r["candidate_entity_ids"]).split(",") if x]
        if len(ids) != len(set(ids)):
            issues.append(f"dupes in candidates for {r['source1_entity_id']}")
        bad = [x for x in ids if not x.startswith(("S2-", "S3-")) or x not in valid]
        if bad:
            issues.append(f"bad candidate ids for {r['source1_entity_id']}: {bad[:3]}")
        cmap[r["source1_entity_id"]] = set(ids)
    for _, r in m.iterrows():
        ids = [x for x in str(r["matched_entity_ids"]).split(",") if x]
        if len(ids) != len(set(ids)):
            issues.append(f"dupes in matches for {r['source1_entity_id']}")
        if not set(ids) <= cmap.get(r["source1_entity_id"], set()):
            issues.append(f"match not in candidates for {r['source1_entity_id']}")
    return issues


issues = check_outputs(OUT_DIR / "matching_results.tsv",
                         OUT_DIR / "candidate_pairs.tsv", set(sample_s1))
print("VALIDATION:", "PASS" if not issues else f"FAIL {issues[:5]}")

# repo validator too, if the student_resource tree is beside the notebook
import subprocess
cands = list(Path.cwd().rglob("utils/validate_submission.py"))
cands += list(Path.cwd().parent.rglob("utils/validate_submission.py"))
if cands:
    print("repo validator found at", cands[0])
else:
    print("(repo validator not beside notebook — inline checks above are the gate)")

## Phase 1 — Normalization check incl. France (runs here)
The tables in the skeleton cell already include the French suffixes/abbreviations (§1.6 of the plan lists why). Quick assertions:

In [ ]:
assert normalize_text("SARL Dupont") == "societe responsabilite limitee dupont"
assert extract_pin("12 Rue de la Paix, 75002 Paris") == "75002"
assert extract_pin("Wadala, Mumbai 400031") == "400031"
assert normalize_text("Lumay Bóral") == normalize_text("Lumay Boral")
print("FR + accent normalization OK")

## Phase 2 — Sample-scale FAISS ANN demo (primary index, per plan)
Dense `IndexFlatIP` over hashed char 3–5g vectors, capped at 15k docs so it fits 4 GB. Reports recall@K against val GT restricted to the pool (denominator = matches present in pool) + mean K + peak RAM.

In [ ]:
import faiss
from sklearn.feature_extraction.text import HashingVectorizer

demo_pool = pool_df.head(POOL_DOCS).reset_index(drop=True)
pool_texts = (demo_pool["norm_name"] + " [SEP] " + demo_pool["norm_addr"]).tolist()
q_texts = [(s1_idx[s].norm_name + " [SEP] " + s1_idx[s].norm_addr) for s in sample_s1]
q_ids = [str(x) for x in demo_pool["entity_id"]]
pool_set = set(q_ids)

vec = HashingVectorizer(analyzer="char_wb", ngram_range=(3, 5), n_features=4096,
                        alternate_sign=False, norm="l2")
t0 = time.time()
X = vec.transform(pool_texts).astype(np.float32).toarray()
index = faiss.IndexFlatIP(X.shape[1])
index.add(X)
Q = vec.transform(q_texts).astype(np.float32).toarray()
D, I = index.search(Q, TOP_K)
dt = time.time() - t0
peak_gb = resource.getrusage(resource.RUSAGE_SELF).ru_maxrss / 1e6
print(f"FAISS demo: pool={len(demo_pool)} dim={X.shape[1]} topK={TOP_K} in {dt:.1f}s, peak-RSS~{peak_gb:.2f} GB")

recalls, covered, ks = [], 0, []
for i, s in enumerate(sample_s1):
    denom = [m for m in val_matches[s] if m in pool_set]
    if not denom:
        continue
    covered += 1
    hits = [q_ids[j] for j in I[i]]
    for k in (1, 5, TOP_K):
        pass
    r = len(set(hits) & set(denom)) / len(denom)
    recalls.append(r)
    ks.append(len(hits))
print(f"pool coverage of sample GT: {covered}/{len(sample_s1)}")
if recalls:
    print(f"recall@{TOP_K} (in-pool denom) mean={np.mean(recalls):.3f}  mean-K={np.mean(ks):.1f}")
print("NOTE: skeleton pool is PIN-biased by construction — treat this as an index smoke test, not a recall claim.")

## Phase 2 scale-up — full data (STUB, needs a bigger instance)
Recipe: per-country FAISS shard (IVF-PQ trained on GPU box, or HNSW on CPU with ≥64 GB RAM), chunked S1 queries (50k/chunk), streaming candidate writes, then the recall gate (≥95% ceiling → minimize mean K per §1.6). `ml.m5.4xlarge` (64 GB) is the minimum sane target; `ml.m5.12xlarge` or a SageMaker Processing job for the full 10M-doc index.

In [ ]:
assert RUN_FULL, (
    "Full-data blocking is disabled: ml.t3.medium (4 GB) cannot index ~10M docs. "
    "Set RUN_FULL=True only on ml.m5.4xlarge or larger, then implement: "
    "(1) per-country shard, (2) FAISS IVF-PQ/HNSW build, (3) chunked query, "
    "(4) PIN/token backfill + rank-merge + adaptive cap, (5) recall/K gate on val.")
# full-scale code lands here (Phase 2 of ENTITY_RESOLUTION_PLAN.md)

## Phase 3 / 4 / 5 — Roadmap (stubs; built after the blocking gate passes)
- **Phase 3:** pairwise features (`suffix_match`, Jaro-Winkler/RapidFuzz, TF-IDF cosine, token Jaccard, number/PIN/city match, `country_match`, S2-vs-S3) → LightGBM baseline (CPU) → `all-MiniLM-L6-v2` bi-encoder fine-tune (needs GPU instance, e.g. `ml.g5.xlarge`) → optional cross-encoder rerank top-10.
- **Phase 4:** tune global `tau` on val macro-F0.5 (expect ~0.6–0.8); `max_score < tau` → empty list.
- **Phase 5:** chunked test inference → both TSVs → `validate_submission.py --check-ids` must PASS → submission zip.

Next step on this instance: extend the skeleton's `pair_score` into a trained LightGBM on val candidates, then re-run the F0.5 + validation cells.

In [ ]:
import json
print(json.dumps({"sample_S1": len(sample_s1), "pool_rows": len(pool_df),
                   "skeleton_F05": round(float(skel_f05), 4),
                   "outputs": sorted(str(p) for p in OUT_DIR.glob('*.tsv'))}, indent=2))